## Environment Setup and Dependency Installation

In [1]:
!pip install torch torchaudio transformers datasets peft bitsandbytes jiwer gradio accelerate
!pip install --upgrade transformers peft torchao accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 31.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:
      Successfully uninstalled transformers-5.10.2


## Import Required Libraries and Initialize Hyperparameters


In [2]:
import os
import torch
import gradio as gr
import torchaudio

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import load_dataset, Audio
from dataclasses import dataclass
from jiwer import wer, cer
from peft import LoraConfig,PeftModel, PeftConfig, get_peft_model
from typing import Any, Dict, List, Union

In [3]:
MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "gmenon/slt-lyrics-audio"
OUTPUT_DIR = "./whisper-lora-autolyrics"
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
MAX_STEPS = 200
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### Load, Preprocess, and Tokenize Audio-Transcription Dataset

In [4]:
print("\n--- Loading Dataset ---")
raw_dataset = load_dataset(DATASET_NAME)

raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="english", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    target_text = batch.get("text", batch.get("lyrics", ""))
    batch["labels"] = processor.tokenizer(target_text).input_ids
    return batch

print("Preprocessing dataset (Resampling & Tokenizing)...")
train_split = raw_dataset["train"].select(range(min(150, len(raw_dataset["train"]))))
test_split = raw_dataset["test"].select(range(min(12, len(raw_dataset["test"])))) if "test" in raw_dataset else train_split

train_dataset = train_split.map(prepare_dataset, remove_columns=train_split.column_names)
eval_dataset = test_split.map(prepare_dataset, remove_columns=test_split.column_names)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


--- Loading Dataset ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/641 [00:00<?, ?B/s]

data/train-00000-of-00012-bdae940d202447(…):   0%|          | 0.00/416M [00:00<?, ?B/s]

data/train-00001-of-00012-959612f33247a9(…):   0%|          | 0.00/445M [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Preprocessing dataset (Resampling & Tokenizing)...


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

###  Initialize Whisper Model and Configure LoRA Adaptation

In [5]:
from transformers import BitsAndBytesConfig


quantization_config = None
if device == "cuda":
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto" if device == "cuda" else None
)

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

## Configure Training Pipeline and Fine-Tune Whisper using LoRA


In [6]:
print("\n--- Loading Model and Applying LoRA ---")

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None
)

model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    )

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


print("\n--- Defining Evaluation Metric ---")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer_score = 100 * wer(label_str, pred_str)
    return {"wer": wer_score}


print("\n--- Setting Up Trainer ---")

train_dataset = train_dataset.select_columns(["input_features", "labels"])
eval_dataset = eval_dataset.select_columns(["input_features", "labels"])

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True if device == "cuda" else False,
    eval_strategy="steps",
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=10,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    label_names=["labels"],
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("\n--- Starting Training ---")
trainer.train()

print("\n--- Saving the Finetuned Adapter ---")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model and processor successfully saved to {OUTPUT_DIR}")


--- Loading Model and Applying LoRA ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429

--- Defining Evaluation Metric ---

--- Setting Up Trainer ---

--- Starting Training ---


Step,Training Loss,Validation Loss,Wer
50,4.382355,2.000602,15000
100,0.711556,0.034650,0
150,0.000175,0.000084,15000
200,0.000151,0.000076,15000


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para


--- Saving the Finetuned Adapter ---
Model and processor successfully saved to ./whisper-lora-autolyrics


## Compare Fine-Tuned Model Against Baseline Whisper Performance


In [7]:

def evaluate_predictions(base_model_name, adapter_path, eval_data):
    base_model = WhisperForConditionalGeneration.from_pretrained(base_model_name).to(device)

    peft_model = WhisperForConditionalGeneration.from_pretrained(base_model_name)
    peft_model = PeftModel.from_pretrained(peft_model, adapter_path).to(device)

    references = []
    base_preds = []
    lora_preds = []

    for sample in eval_data.select(range(min(5, len(eval_data)))):
        input_feats = torch.tensor([sample["input_features"]]).to(device)

        ref_ids = [i for i in sample["labels"] if i != -100]
        ref_text = processor.tokenizer.decode(ref_ids, skip_special_tokens=True)
        references.append(ref_text)

        with torch.no_grad():
            base_gen = base_model.generate(input_features=input_feats)
            base_pred = processor.tokenizer.decode(base_gen[0], skip_special_tokens=True)
            base_preds.append(base_pred)

            lora_gen = peft_model.generate(input_features=input_feats)
            lora_pred = processor.tokenizer.decode(lora_gen[0], skip_special_tokens=True)
            lora_preds.append(lora_pred)
    base_wer = wer(references, base_preds)
    lora_wer = wer(references, lora_preds)
    base_cer = cer(references, base_preds)
    lora_cer = cer(references, lora_preds)

    print("Results")
    print(f"Base Model ({base_model_name}) :- WER: {base_wer:.4f} %| CER: {base_cer:.4f} %")
    print(f"LoRA Fine-Tuned Model     :- WER: {lora_wer:.4f} %| CER: {lora_cer:.4f} %")
    if base_wer > 0:
        improvement = ((base_wer - lora_wer) / base_wer) * 100
        print(f"Relative WER Reduction: {improvement:.2f}%")


evaluate_predictions(MODEL_NAME, OUTPUT_DIR, eval_dataset)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Results
Base Model (openai/whisper-small) :- WER: 28.0000 %| CER: 127.0000 %
LoRA Fine-Tuned Model     :- WER: 0.0000 %| CER: 0.0000 %
Relative WER Reduction: 100.00%


## Deploy Interactive Lyrics Transcription Web Application Using Gradio

In [ ]:
print("\n--- Launching Gradio Demo ---")

import gradio as gr
import torch
import librosa
from transformers import WhisperForConditionalGeneration

# # Load base model
# base_model = WhisperForConditionalGeneration.from_pretrained(
#     MODEL_NAME
# )

# # Load trained LoRA adapter
# demo_model = PeftModel.from_pretrained(
#     base_model,
#     OUTPUT_DIR
# )

demo_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
).to(device)

demo_model.eval()

print("AutoLyrics model loaded successfully!")


def transcribe_audio(audio_file):

    if audio_file is None:
        return "No audio uploaded.", "No audio uploaded."

    speech_array, _ = librosa.load(
        audio_file,
        sr=16000,
        mono=True
    )

    input_features = processor.feature_extractor(
        speech_array,
        sampling_rate=16000
    ).input_features[0]

    input_tensor = torch.tensor(
        [input_features],
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():

        predicted_ids = demo_model.generate(
            input_features=input_tensor,
            max_new_tokens=225,
            num_beams=3
        )

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return transcription, transcription


# GUI
theme = gr.themes.Soft()

with gr.Blocks(theme=theme) as demo:

    gr.Markdown("""
    # AutoLyrics

    ### Singing Lyrics Transcription using Fine-Tuned Whisper

    Upload or record a song and generate lyrics automatically.

    Developed By: Apoorva, Anwesha, Sujal
    """)

    with gr.Accordion("Model Information", open=False):

        gr.Markdown("""
        **Model:** Fine-Tuned OpenAI Whisper

        **Task:** Lyrics Transcription from Singing Audio

        **Sampling Rate:** 16 kHz

        **Decoding:** Beam Search (num_beams = 3)

        **Output:** Predicted Song Lyrics
        """)

    with gr.Row():

        with gr.Column(scale=1):

            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="Upload or Record Audio"
            )

            transcribe_btn = gr.Button(
                "Generate Lyrics",
                variant="primary"
            )

        with gr.Column(scale=2):

            model_output = gr.Textbox(
                label="Generated Lyrics",
                lines=15,
                show_copy_button=True
            )

            reference_output = gr.Textbox(
                label="Reference Lyrics",
                lines=15,
                show_copy_button=True
            )

    transcribe_btn.click(
        fn=transcribe_audio,
        inputs=audio_input,
        outputs=[
            model_output,
            reference_output
        ]
    )

    gr.Markdown("""
    ---
    Built using Whisper, PyTorch, Hugging Face Transformers, and Gradio.
    """)

demo.launch(
    share=True,
    debug=True
)



--- Launching Gradio Demo ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

AutoLyrics model loaded successfully!


/tmp/ipykernel_422/3038874025.py:68: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9f1ff2ea55d7b6f20c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
